# GIAI ĐOẠN 4B: TRÍCH XUẤT ĐẶC TRƯNG RA MA TRẬN SỐ (FEATURE EXTRACTION - CODE TAY CƠ BẢN)

## 1. Nhiệm vụ của thành viên phụ trách Feature
- **Mục tiêu**: Đọc các tập dữ liệu đã qua tiền xử lý tại `data/tien_xu_ly/` và chuyển đổi văn bản thành các ma trận số.
- **Phong cách triển khai**: **Code tay từ đầu (from scratch) theo cách cơ bản, mộc mạc và dễ hiểu nhất**:
  1. **Bag of Words (BoW)**: Dùng `set`, `dict` và vòng lặp `for` để đếm số lần xuất hiện của từ.
  2. **TF-IDF**: Tính trực tiếp $\text{TF} = \frac{\text{Số lần từ xuất hiện}}{\text{Tổng số từ trong câu}}$ và $\text{IDF} = \ln\left(\frac{N + 1}{\text{DF} + 1}\right) + 1$.
  3. **Word2Vec (Skip-gram cơ bản)**: Mạng nơ-ron 1 lớp ẩn đơn giản dự đoán từ ngữ cảnh, tính vector câu bằng trung bình cộng các vector từ.
- **Nguyên tắc học máy (Bắt buộc)**:
  - Mô hình **CHỈ ĐƯỢC HỌC (FIT) TRÊN TẬP TRAIN**.
  - Tập **Validation** và **Test** chỉ được dùng phương thức `transform` theo từ điển đã học từ Train.
- **Cấu trúc lưu trữ đầu ra**:
  - Lưu toàn bộ mô hình và dữ liệu đặc trưng vào thư mục riêng biệt: `models/features/`
    + `models/features/tfidf_vectorizer.pkl`
    + `models/features/bow_vectorizer.pkl`
    + `models/features/word2vec_model.pkl`
    + `models/features/feature_data.joblib`

### Bước 1: Khai báo thư viện cơ bản và nạp dữ liệu
- Nạp 3 tập dữ liệu `train.csv`, `validation.csv`, `test.csv` từ `data/tien_xu_ly/` do Giai đoạn 4A sinh ra.
- Tách riêng danh sách câu văn bản (`X`) và nhãn cảm xúc (`y`).

In [ ]:
import os
import math
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
try:
    display
except NameError:
    from IPython.display import display

# Thiết lập thư mục dữ liệu và thư mục lưu model đặc trưng riêng (models/features/)
DATA_DIR = Path("../data/tien_xu_ly")
MODEL_FEATURE_DIR = Path("../models/features")
MODEL_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Đọc 3 tập dữ liệu đã qua tiền xử lý sâu từ Giai đoạn 4A
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val   = pd.read_csv(DATA_DIR / "validation.csv")
df_test  = pd.read_csv(DATA_DIR / "test.csv")

# Xử lý chuỗi rỗng / NaN nếu có
df_train["van_ban"] = df_train["van_ban"].fillna("").astype(str)
df_val["van_ban"]   = df_val["van_ban"].fillna("").astype(str)
df_test["van_ban"]  = df_test["van_ban"].fillna("").astype(str)

# Tách riêng danh sách câu văn bản (X) và mảng nhãn cảm xúc (y)
X_train_text = df_train["van_ban"].tolist()
y_train      = df_train["label"].to_numpy()

X_val_text   = df_val["van_ban"].tolist()
y_val        = df_val["label"].to_numpy()

X_test_text  = df_test["van_ban"].tolist()
y_test       = df_test["label"].to_numpy()

print("="*60)
print("THỐNG KÊ DỮ LIỆU ĐẦU VÀO TỪ data/tien_xu_ly/:")
print(f"- 1. Tập Train      : {len(X_train_text):<5} câu | Phân bố nhãn: {dict(pd.Series(y_train).value_counts())}")
print(f"- 2. Tập Validation : {len(X_val_text):<5} câu | Phân bố nhãn: {dict(pd.Series(y_val).value_counts())}")
print(f"- 3. Tập Test       : {len(X_test_text):<5} câu | Phân bố nhãn: {dict(pd.Series(y_test).value_counts())}")
print("="*60)

### Bước 2: Tự viết tay 3 phương pháp trích chọn đặc trưng

#### 2.1. Bag of Words (BoW) cơ bản
- **Thuật toán**:
  1. `fit`: Duyệt qua tất cả các câu trong tập Train, gom các từ duy nhất vào một `set` -> tạo danh sách từ vựng `vocab` và bảng ánh xạ `word2idx`.
  2. `transform`: Với mỗi câu, tạo vector toàn số 0 có độ dài bằng kích thước từ điển. Đếm số lần mỗi từ trong từ điển xuất hiện trong câu đó.

In [ ]:
class BoW_CoBan:
    """
    Mô hình Bag of Words tự code tay đơn giản bằng Python cơ bản:
    - fit: Tìm tập từ vựng duy nhất từ tập Train.
    - transform: Đếm số lần xuất hiện của từng từ trong câu.
    """
    def __init__(self):
        self.vocab = []         # Danh sách các từ duy nhất
        self.word2idx = {}      # Từ điển ánh xạ: {từ: vị trí cột}

    def fit(self, danh_sach_cau: list[str]):
        # 1. Thu thập tất cả các từ xuất hiện trong tập Train
        tap_tu = set()
        for cau in danh_sach_cau:
            for tu in cau.split():
                tap_tu.add(tu)
        
        # 2. Sắp xếp từ vựng theo bảng chữ cái để cố định vị trí
        self.vocab = sorted(list(tap_tu))
        self.word2idx = {tu: i for i, tu in enumerate(self.vocab)}
        return self

    def transform(self, danh_sach_cau: list[str]) -> np.ndarray:
        # 3. Tạo ma trận đếm số lần xuất hiện của từng từ trong câu
        so_tu_vung = len(self.vocab)
        ma_tran = []

        for cau in danh_sach_cau:
            vector = [0] * so_tu_vung
            cac_tu = cau.split()
            for tu in cac_tu:
                if tu in self.word2idx:      # Nếu từ có trong từ điển Train
                    cot = self.word2idx[tu]
                    vector[cot] += 1
            ma_tran.append(vector)

        return np.array(ma_tran)

    def fit_transform(self, danh_sach_cau: list[str]) -> np.ndarray:
        return self.fit(danh_sach_cau).transform(danh_sach_cau)

    def get_feature_names_out(self):
        return np.array(self.vocab)

#### 2.2. TF-IDF cơ bản
- **Thuật toán**:
  1. $\text{TF}(t, d) = \frac{\text{Số lần từ } t \text{ xuất hiện trong câu } d}{\text{Tổng số từ trong câu } d}$
  2. $\text{IDF}(t) = \ln\left(\frac{N + 1}{\text{DF}(t) + 1}\right) + 1$ (với $N$ là tổng số câu của tập Train, $\text{DF}(t)$ là số câu chứa từ $t$).
  3. $\text{TF-IDF} = \text{TF} \times \text{IDF}$.

In [ ]:
class TFIDF_CoBan:
    """
    Mô hình TF-IDF tự code tay đơn giản bằng vòng lặp và công thức chuẩn:
    - fit: Học từ điển và tính giá trị IDF cho từng từ trên tập Train.
    - transform: Tính TF và nhân với IDF để tạo ma trận đặc trưng.
    """
    def __init__(self):
        self.vocab = []         # Danh sách từ vựng
        self.word2idx = {}      # {từ: vị trí cột}
        self.idf = {}           # {từ: trọng số idf}

    def fit(self, danh_sach_cau: list[str]):
        N = len(danh_sach_cau)  # Tổng số câu trong tập Train

        # 1. Tìm tập từ vựng duy nhất từ tập Train
        tap_tu = set()
        for cau in danh_sach_cau:
            for tu in cau.split():
                tap_tu.add(tu)
        self.vocab = sorted(list(tap_tu))
        self.word2idx = {tu: i for i, tu in enumerate(self.vocab)}

        # 2. Đếm số câu chứa từng từ (Document Frequency - DF)
        for tu in self.vocab:
            so_cau_chua_tu = sum(1 for cau in danh_sach_cau if tu in cau.split())
            # Tính IDF với công thức làm mượt chuẩn (Smooth IDF)
            self.idf[tu] = math.log((N + 1) / (so_cau_chua_tu + 1)) + 1.0
        return self

    def transform(self, danh_sach_cau: list[str]) -> np.ndarray:
        so_tu_vung = len(self.vocab)
        ma_tran = []

        for cau in danh_sach_cau:
            cac_tu = cau.split()
            tong_so_tu = len(cac_tu)
            vector = [0.0] * so_tu_vung

            if tong_so_tu > 0:
                # Đếm số lần mỗi từ xuất hiện trong câu này
                dem_tu = {}
                for tu in cac_tu:
                    dem_tu[tu] = dem_tu.get(tu, 0) + 1

                # Tính TF và TF-IDF
                for tu, so_lan in dem_tu.items():
                    if tu in self.word2idx:  # Chỉ tính các từ đã học từ Train
                        tf = so_lan / tong_so_tu
                        cot = self.word2idx[tu]
                        vector[cot] = tf * self.idf[tu]

            ma_tran.append(vector)

        return np.array(ma_tran)

    def fit_transform(self, danh_sach_cau: list[str]) -> np.ndarray:
        return self.fit(danh_sach_cau).transform(danh_sach_cau)

    def get_feature_names_out(self):
        return np.array(self.vocab)

#### 2.3. Word2Vec cơ bản (Skip-gram)
- **Thuật toán**:
  1. Tạo các cặp huấn luyện `(từ_gốc, từ_lân_cận)` trong phạm vi cửa sổ `window_size`.
  2. Lan truyền tiến (Forward): Nhúng từ qua ma trận trọng số $W_1$, tính điểm qua ma trận $W_2$ và qua hàm Softmax để dự đoán từ ngữ cảnh.
  3. Lan truyền ngược (Backward): Cập nhật $W_1, W_2$ bằng sai số Gradient Descent.
  4. Biểu diễn câu: Lấy trung bình cộng vector của tất cả các từ trong câu.

In [ ]:
class Word2Vec_CoBan:
    """
    Mô hình Word2Vec (Skip-gram) tự code tay đơn giản bằng NumPy:
    - fit: Học ma trận vector nhúng từ (W1) từ tập Train.
    - transform: Biến đổi mỗi câu thành vector bằng trung bình cộng các từ trong câu.
    """
    def __init__(self, vector_size=20, window_size=2, lr=0.05, epochs=15):
        self.vector_size = vector_size  # Chiều của vector nhúng (VD: 20 hoặc 30)
        self.window_size = window_size  # Bán kính cửa sổ ngữ cảnh
        self.lr = lr                    # Tốc độ học (learning rate)
        self.epochs = epochs            # Số vòng lặp huấn luyện
        self.vocab = []
        self.word2idx = {}
        self.W1 = None                  # Ma trận vector từ (V x vector_size)
        self.W2 = None                  # Ma trận phân loại (vector_size x V)

    def _softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum()

    def fit(self, danh_sach_cau: list[str]):
        # 1. Tìm tập từ vựng duy nhất
        tap_tu = set()
        for cau in danh_sach_cau:
            for tu in cau.split():
                tap_tu.add(tu)
        self.vocab = sorted(list(tap_tu))
        self.word2idx = {tu: i for i, tu in enumerate(self.vocab)}
        V = len(self.vocab)
        if V == 0:
            return self

        # 2. Khởi tạo 2 ma trận trọng số ngẫu nhiên
        np.random.seed(42)
        self.W1 = np.random.uniform(-0.5, 0.5, (V, self.vector_size))
        self.W2 = np.random.uniform(-0.5, 0.5, (self.vector_size, V))

        # 3. Tạo các cặp từ (từ_gốc, từ_ngữ_cảnh)
        cac_cap = []
        for cau in danh_sach_cau:
            cac_tu = cau.split()
            n = len(cac_tu)
            for i in range(n):
                idx_goc = self.word2idx[cac_tu[i]]
                for j in range(max(0, i - self.window_size), min(n, i + self.window_size + 1)):
                    if i != j:
                        cac_cap.append((idx_goc, self.word2idx[cac_tu[j]]))

        # 4. Vòng lặp huấn luyện Skip-gram cơ bản
        for epoch in range(self.epochs):
            for center_idx, context_idx in cac_cap:
                # Lan truyền tiến (Forward)
                h = self.W1[center_idx]        # Vector nhúng của từ trung tâm
                u = np.dot(h, self.W2)          # Điểm dự đoán (V,)
                y_pred = self._softmax(u)

                # Sai số (Error)
                e = y_pred.copy()
                e[context_idx] -= 1             # Trừ 1 tại từ ngữ cảnh đúng

                # Lan truyền ngược (Backward - Gradient)
                dW2 = np.outer(h, e)
                dW1 = np.dot(self.W2, e)

                # Cập nhật trọng số
                self.W2 -= self.lr * dW2
                self.W1[center_idx] -= self.lr * dW1

        return self

    def transform(self, danh_sach_cau: list[str]) -> np.ndarray:
        # Biến đổi câu thành vector bằng trung bình cộng vector các từ trong câu
        ma_tran_cau = []
        for cau in danh_sach_cau:
            cac_tu = [tu for tu in cau.split() if tu in self.word2idx]
            if len(cac_tu) > 0:
                vector_cac_tu = [self.W1[self.word2idx[tu]] for tu in cac_tu]
                vector_cau = np.mean(vector_cac_tu, axis=0)
            else:
                vector_cau = np.zeros(self.vector_size)
            ma_tran_cau.append(vector_cau)

        return np.array(ma_tran_cau)

    def most_similar(self, tu: str, topn=3):
        """Tìm các từ có vector gần nhất bằng Cosine Similarity"""
        if tu not in self.word2idx or self.W1 is None:
            return []
        idx = self.word2idx[tu]
        v = self.W1[idx]
        
        # Tính Cosine Similarity
        norms = np.linalg.norm(self.W1, axis=1) + 1e-9
        v_norm = np.linalg.norm(v) + 1e-9
        sims = np.dot(self.W1, v) / (norms * v_norm)
        
        top_idx = sims.argsort()[::-1]
        ket_qua = []
        for i in top_idx:
            if i != idx:
                ket_qua.append((self.vocab[i], round(float(sims[i]), 4)))
                if len(ket_qua) >= topn:
                    break
        return ket_qua

### Bước 3: Huấn luyện trên tập Train và biến đổi trên Validation/Test

> **Quy tắc vàng học máy**: Chỉ gọi `.fit()` trên tập **Train**. Tập **Validation** và **Test** chỉ dùng `.transform()`.

In [ ]:
# ==============================================================================
# 1. CHẠY MÔ HÌNH BAG OF WORDS (BoW)
# ==============================================================================
print("=== 1. CHẠY MÔ HÌNH BAG OF WORDS (BoW) ===")
bow_model = BoW_CoBan()
X_train_bow = bow_model.fit_transform(X_train_text)  # CHỈ FIT TRÊN TRAIN
X_val_bow   = bow_model.transform(X_val_text)        # CHỈ TRANSFORM
X_test_bow  = bow_model.transform(X_test_text)       # CHỈ TRANSFORM

print(f"-> Kích thước từ vựng BoW: {len(bow_model.vocab)} từ")
print(f"-> X_train_bow shape: {X_train_bow.shape}")
print(f"-> X_val_bow shape  : {X_val_bow.shape}")
print(f"-> X_test_bow shape : {X_test_bow.shape}")

# ==============================================================================
# 2. CHẠY MÔ HÌNH TF-IDF (Phương pháp chính)
# ==============================================================================
print("\n=== 2. CHẠY MÔ HÌNH TF-IDF ===")
tfidf_model = TFIDF_CoBan()
X_train_tfidf = tfidf_model.fit_transform(X_train_text)  # CHỈ FIT TRÊN TRAIN
X_val_tfidf   = tfidf_model.transform(X_val_text)        # CHỈ TRANSFORM
X_test_tfidf  = tfidf_model.transform(X_test_text)       # CHỈ TRANSFORM

print(f"-> Kích thước từ vựng TF-IDF: {len(tfidf_model.vocab)} từ")
print(f"-> X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"-> X_val_tfidf shape  : {X_val_tfidf.shape}")
print(f"-> X_test_tfidf shape : {X_test_tfidf.shape}")

# ==============================================================================
# 3. CHẠY MÔ HÌNH WORD2VEC (Skip-gram cơ bản)
# ==============================================================================
print("\n=== 3. CHẠY MÔ HÌNH WORD2VEC ===")
w2v_model = Word2Vec_CoBan(vector_size=20, window_size=2, lr=0.05, epochs=15)
w2v_model.fit(X_train_text)                           # CHỈ FIT TRÊN TRAIN
X_train_w2v = w2v_model.transform(X_train_text)       # VECTOR CÂU TRAIN
X_val_w2v   = w2v_model.transform(X_val_text)         # VECTOR CÂU VAL
X_test_w2v  = w2v_model.transform(X_test_text)        # VECTOR CÂU TEST

print(f"-> Chiều vector Word2Vec: {w2v_model.vector_size}")
print(f"-> X_train_w2v shape: {X_train_w2v.shape}")
print(f"-> X_val_w2v shape  : {X_val_w2v.shape}")
print(f"-> X_test_w2v shape : {X_test_w2v.shape}")

### Bước 4: Kiểm tra và phân tích đặc trưng trích xuất được
- Xem Top các từ có trọng số TF-IDF trung bình cao nhất trên tập Train.
- Xem thử độ tương đồng của từ trong mô hình Word2Vec.

In [ ]:
# 1. Thống kê Top 10 từ có điểm TF-IDF trung bình cao nhất tập Train
mean_tfidf = X_train_tfidf.mean(axis=0)
top_idx = mean_tfidf.argsort()[::-1][:10]

df_top_tfidf = pd.DataFrame({
    "Thứ hạng": range(1, len(top_idx) + 1),
    "Từ vựng": [tfidf_model.vocab[i] for i in top_idx],
    "Điểm TF-IDF TB": np.round(mean_tfidf[top_idx], 4)
})
print("=== TOP TỪ CÓ TRỌNG SỐ TF-IDF CAO NHẤT TẬP TRAIN ===")
display(df_top_tfidf)

# 2. Kiểm tra từ tương đồng ngữ nghĩa trong Word2Vec
print("\n=== KIỂM TRA ĐỘ TƯƠNG ĐỒNG WORD2VEC (Cosine Similarity) ===")
cac_tu_kiem_tra = [tu for tu in ["tốt", "chất_lượng", "giao", "tệ", "thất_vọng"] if tu in w2v_model.word2idx]
for tu in cac_tu_kiem_tra[:3]:
    print(f"-> Từ '{tu}' có các từ tương đồng nhất là: {w2v_model.most_similar(tu, topn=3)}")

### Bước 5: Lưu trữ mô hình đặc trưng vào thư mục riêng: `models/features/`
- Lưu toàn bộ các vectorizer và ma trận đặc trưng vào `models/features/` để tách biệt rõ ràng với các mô hình học máy phân loại.

In [ ]:
# 1. Lưu TF-IDF Vectorizer tự viết vào thư mục models/features/
tfidf_path = MODEL_FEATURE_DIR / "tfidf_vectorizer.pkl"
joblib.dump(tfidf_model, tfidf_path)
print(f"[1/4] Đã lưu TF-IDF Model tại       : {tfidf_path}")

# 2. Lưu BoW Vectorizer tự viết vào thư mục models/features/
bow_path = MODEL_FEATURE_DIR / "bow_vectorizer.pkl"
joblib.dump(bow_model, bow_path)
print(f"[2/4] Đã lưu BoW Model tại          : {bow_path}")

# 3. Lưu Word2Vec Model tự viết vào thư mục models/features/
w2v_path = MODEL_FEATURE_DIR / "word2vec_model.pkl"
joblib.dump(w2v_model, w2v_path)
print(f"[3/4] Đã lưu Word2Vec Model tại     : {w2v_path}")

# 4. Đóng gói toàn bộ ma trận đặc trưng vào thư mục models/features/
bundle_path = MODEL_FEATURE_DIR / "feature_data.joblib"
feature_bundle = {
    "tfidf": {
        "X_train": X_train_tfidf,
        "X_val": X_val_tfidf,
        "X_test": X_test_tfidf,
        "vocab": tfidf_model.vocab
    },
    "bow": {
        "X_train": X_train_bow,
        "X_val": X_val_bow,
        "X_test": X_test_bow,
        "vocab": bow_model.vocab
    },
    "word2vec": {
        "X_train": X_train_w2v,
        "X_val": X_val_w2v,
        "X_test": X_test_w2v,
        "vector_size": w2v_model.vector_size
    },
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test
}
joblib.dump(feature_bundle, bundle_path)
print(f"[4/4] Đã lưu Bundle ma trận đặc trưng: {bundle_path}")

print("\n" + "="*65)
print("HOÀN THÀNH GIAI ĐOẠN 4B (FEATURE EXTRACTION):")
print("- Đã lưu toàn bộ model trích chọn đặc trưng vào: models/features/")
print("="*65)